In [0]:
%sql
USE CATALOG cursodataacademy;
USE SCHEMA hotel_db;

(CREATE / INSERT) El equipo comercial quiere hacer un experimento rápido. Crea una tabla temporal (tb_reservas_prueba) y usa el comando INSERT con SELECT para insertar únicamente las reservas que corresponden a 'City Hotel' y que fueron canceladas (is_canceled = 1).


In [0]:
%sql
CREATE OR REPLACE TABLE tb_reservas_prueba AS
SELECT * FROM tb_reservas WHERE 1 = 0;

In [0]:
%sql
INSERT INTO tb_reservas_prueba
SELECT * FROM tb_reservas
WHERE hotel = 'City Hotel' AND is_canceled = 1;

Conclusión: Se aisló en una tabla de prueba el segmento de City Hotel cancelado, para experimentar sin tocar los datos reales. Sirve de base para medir la fuga por cancelaciones en ese hotel.

(UPDATE) Se detectó un error en los sistemas de captura de datos: los huéspedes que registraron babies en realidad no tenían bebés, sino que era un error del canal. Actualiza la tabla tb_reservas_prueba y cambia el valor de babies a 0 para todos los registros.


In [0]:
%sql
UPDATE tb_reservas_prueba
SET babies = 0;

Conclusión: Se corrigió el error de captura solo en la tabla de prueba. La tabla original queda intacta, lo que protege los reportes oficiales.

(CAST / CONCAT) Necesitamos tener la fecha de llegada en un formato legible. Selecciona las columnas de año, mes (que viene en texto, ej. 'July') y día, y concaténalas en una nueva columna llamada arrival_date_string con el formato "DD-Mes-YYYY" (Ej: "25-July-2015").


In [0]:
%sql
SELECT arrival_date_year, arrival_date_month, arrival_date_day_of_month,
       CONCAT(LPAD(arrival_date_day_of_month, 2, '0'), '-', arrival_date_month, '-', arrival_date_year) AS arrival_date_string
FROM tb_reservas;

Conclusión: La fecha de llegada queda legible en formato DD-Mes-YYYY para reportes. Al ser texto no sirve para cálculos, por eso se conserva también arrival_date como fecha.

(LIKE) Queremos filtrar a un segmento muy específico del canal de distribución. Selecciona todas las reservas donde distribution_channel empiece con la letra 'T'.


In [0]:
%sql
SELECT *
FROM tb_reservas
WHERE distribution_channel LIKE 'T%';

Conclusión: El filtro devuelve el canal TA/TO (agencias de viaje y marketing), que es el que cobra USD 1.5 por reserva. Es el segmento central del problema.

(COUNT / SUM / AVG) Calcula a nivel global para todo el hotel: la cantidad total de reservas, la suma total de solicitudes especiales (total_of_special_requests) y el promedio del precio por noche (adr).


In [0]:
%sql
SELECT COUNT(*) AS total_reservas,
       SUM(total_of_special_requests) AS total_solicitudes_especiales,
       AVG(adr) AS adr_promedio
FROM tb_reservas;

Conclusión: Define la línea base del negocio: volumen de reservas, solicitudes especiales y tarifa promedio por noche, contra la que se comparan los segmentos siguientes.

(DELETE) En tu tabla de pruebas (tb_reservas_prueba), elimina todas las reservas que tuvieron un precio por noche (adr) igual a 0.0, ya que se consideran pruebas del sistema y no reservas reales.


In [0]:
%sql
DELETE FROM tb_reservas_prueba
WHERE adr = 0.0;

Conclusión: Se eliminan las reservas con tarifa cero, que son pruebas del sistema y distorsionan promedios y ingresos.

(GROUP BY) El gerente quiere saber cuántas reservas entraron por cada canal de distribución. Obtén la cantidad de reservas agrupadas por distribution_channel.


In [0]:
%sql
SELECT distribution_channel, COUNT(*) AS cantidad_reservas
FROM tb_reservas
GROUP BY distribution_channel
ORDER BY cantidad_reservas DESC;

Conclusión: Muestra qué canales concentran las reservas. Si TA/TO domina, la exposición al pago por reserva, aunque se cancele, es alta y justifica renegociar el contrato.

(CASE WHEN / Tipos de Datos) Crea una consulta que clasifique las reservas en base al precio (adr): Si es menor a 50 es "Económico", si está entre 50 y 150 es "Estándar", y mayor a 150 es "Premium".

In [0]:
%sql
SELECT adr,
       CASE WHEN adr < 50 THEN 'Económico'
            WHEN adr BETWEEN 50 AND 150 THEN 'Estándar'
            ELSE 'Premium' END AS categoria_precio
FROM tb_reservas;

Conclusión: Segmenta cada reserva por nivel de precio por noche, lo que permite medir luego si la cancelación depende del rango tarifario.

(Funciones de Fecha - DATE_SUB / DATE_ADD) (Considera que existe una columna llamada arrival_date de tipo DATE para este ejercicio). Sabemos el lead_time (días de anticipación). Usa la función de Databricks para restar los días de lead_time a la arrival_date y así obtener la fecha exacta en la que el cliente hizo la reservación. Llámala booking_date.


In [0]:
%sql
SELECT arrival_date, lead_time,
       DATE_SUB(arrival_date, CAST(lead_time AS INT)) AS booking_date
FROM tb_reservas;

Conclusión: Se reconstruye la fecha real en que el cliente reservó. Permite estudiar la estacionalidad de la compra y cuánto tiempo antes se cancela.


(Funciones Matemáticas) Queremos calcular una "cuota de riesgo". Selecciona el lead_time, divídelo entre 30 (para obtener meses aproximados) y usa la función CEIL() (redondeo hacia arriba) para mostrar el resultado en una columna llamada meses_anticipacion_techo.


In [0]:
%sql
SELECT lead_time,
       CEIL(lead_time / 30) AS meses_anticipacion_techo
FROM tb_reservas;

Conclusión: Convierte los días de anticipación en meses, redondeando hacia arriba, lo que facilita agrupar las reservas por antelación.